# Vesuvius Challenge 2025 - V3 nnUNet-Style Training

## Matching Host Baseline Architecture (0.543 raw, 0.562 w/ post-processing)

### Key Changes from V2:
| Component | V2 (0.531 LB) | V3 (This) |
|-----------|---------------|------------|
| Norm | GroupNorm | **InstanceNorm3d (affine=True)** |
| Conv bias | False | **True** |
| Loss | Dice + BCE (sigmoid) | **Dice + CE (softmax, per-sample)** |
| Skeleton | 3D soft skeleton | **MedialSurfaceRecall (2D slices)** |
| Augmentation | Flips + rot90 + intensity | **Full nnUNet pipeline (12+ transforms)** |
| Patch size | 192 | **128** (faster epochs) |
| Decoder | ConvBlock | **Plain Conv + InstanceNorm** |

### References:
- [nnUNet v2](https://github.com/MIC-DKFZ/nnUNet)
- [Skeleton Recall Loss (ECCV 2024)](https://github.com/MIC-DKFZ/Skeleton-Recall)
- [soft-clDice (CVPR 2021)](https://github.com/jocpae/clDice)

In [ ]:
!pip install imagecodecs connected-components-3d tifffile -q
!pip install monai[cucim] -q

In [ ]:
# =============================================================================
# CELL 1: IMPORTS & OPTIMIZED CONFIGURATION FOR 3-MIN EPOCHS
# =============================================================================

import os
import gc
import json
import math
import random
import warnings
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from scipy import ndimage
from scipy.ndimage import distance_transform_edt, gaussian_filter
from skimage.morphology import skeletonize
import multiprocessing
try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False

warnings.filterwarnings('ignore')

# =============================================================================
# OPTIMIZED CONFIGURATION FOR 3-MIN EPOCHS (MAXIMIZE RESULT)
# =============================================================================

@dataclass
class Config:
    # Data paths
    DATA_ROOT: Path = Path("/kaggle/input/3d-volume-training-data")
    CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints_v3")
    
    # Model architecture (balanced for speed and capacity)
    PATCH_SIZE: Tuple[int, int, int] = (128, 128, 128)  # Standard size, fits well in memory
    FEATURES: List[int] = None  # [32, 64, 128, 256, 320, 320] - Standard nnUNet sizes
    N_BLOCKS: List[int] = None  # [1, 3, 4, 6, 6, 6] - Standard nnUNet depths
    USE_SCSE: bool = True       # Keep attention for better representation
    USE_DEEP_SUPERVISION: bool = True
    
    # Training settings (optimized for 3-min epochs)
    EPOCHS: int = 1200
    BATCH_SIZE: int = 16  # CRITICAL: Increased from 2 to 8 for H100 throughput
    NUM_WORKERS: int = 12 # Increased for data loading parallelism
    LEARNING_RATE: float = 0.004  # Increased from 0.01 (sqrt(8/2) * 0.01)
    MOMENTUM: float = 0.99
    WEIGHT_DECAY: float = 3e-5
    GRADIENT_CLIP: float = 12.0
    
    # Loss weights (focus on core losses, moderate topology)
    DICE_WEIGHT: float = 0.5       # Core segmentation loss
    CE_WEIGHT: float = 0.5         # Core segmentation loss
    SKELETON_WEIGHT: float = 0.25  # Reduced from 0.3 to balance speed/result
    CLDICE_WEIGHT: float = 0.15    # Reduced from 0.2 to balance speed/result
    
    # Loss scheduling (starts topology losses reasonably early)
    SKELETON_START_EPOCH: int = 150    # Earlier than default (200) but not immediate
    SKELETON_WARMUP_EPOCHS: int = 150
    CLDICE_START_EPOCH: int = 300      # Earlier than default (400) but not immediate
    CLDICE_WARMUP_EPOCHS: int = 150
    
    # Training control
    RESUME_TRAINING: bool = False
    VALIDATE_EVERY: int = 10  # Less frequent validation saves time
    SAVE_EVERY: int = 20      # Less frequent saving saves time
    OVERRIDE_LR: float = 0.0
    
    # Device
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True  # Crucial for speed on H100
    
    # Data (standard settings, full fraction for best result)
    DATA_FRACTION: float = 1.0 # Use all available data
    PATCHES_PER_VOLUME: int = 16  # Increased to get more samples per epoch with larger batch
    PRELOAD_ALL: bool = True      # Preload for maximum speed
    FG_OVERSAMPLE_RATIO: float = 0.33  # nnUNet default, helps with rare class
    
    # Augmentation (full nnUNet pipeline, crucial for generalization/result quality)
    # Keep all probabilities as per original notebook for maximum benefit
    AUG_ROTATION_RANGE: float = 30.0
    AUG_ROTATION_P: float = 0.2
    AUG_SCALE_RANGE: Tuple[float, float] = (0.7, 1.4)
    AUG_SCALE_P: float = 0.2
    AUG_GAUSSIAN_NOISE_P: float = 0.1
    AUG_GAUSSIAN_NOISE_STD: Tuple[float, float] = (0.0, 0.1)
    AUG_GAUSSIAN_BLUR_P: float = 0.2
    AUG_GAUSSIAN_BLUR_SIGMA: Tuple[float, float] = (0.5, 1.0)
    AUG_BRIGHTNESS_P: float = 0.15
    AUG_BRIGHTNESS_RANGE: float = 0.3
    AUG_CONTRAST_P: float = 0.15
    AUG_CONTRAST_RANGE: Tuple[float, float] = (0.75, 1.25)
    AUG_GAMMA_P: float = 0.3
    AUG_GAMMA_RANGE: Tuple[float, float] = (0.7, 1.5)
    AUG_GAMMA_INVERT_P: float = 0.1
    AUG_LOW_RES_P: float = 0.25
    AUG_LOW_RES_ZOOM: Tuple[float, float] = (0.5, 1.0)
    AUG_MIRROR_P: float = 0.5
    
    SEED: int = 42
    
    def post_init(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320, 320] # Standard sizes
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 3, 4, 6, 6, 6] # Standard depths
        self.CHECKPOINT_DIR = Path(self.CHECKPOINT_DIR)
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

cfg = Config()
cfg.post_init()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True # Crucial for speed optimization

set_seed(cfg.SEED)

print("="*60)
print("V3 OPTIMIZED CONFIGURATION (3-MIN EPOCHS, MAXIMIZE RESULT)")
print("="*60)
print(f"Patch size: {cfg.PATCH_SIZE}")
print(f"Batch size: {cfg.BATCH_SIZE}")  # CRITICAL CHANGE
print(f"Model: Standard nnUNet sizes and depths")
print(f"Data fraction: {cfg.DATA_FRACTION}")
print(f"Learning Rate: {cfg.LEARNING_RATE}") # SCALED UP
print(f"Augmentation: Full nnUNet pipeline")
print("="*60)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# =============================================================================
# CELL 2: CHECKPOINT MANAGER
# =============================================================================

class CheckpointManager:
    """Manages saving and loading of training checkpoints."""
    
    def __init__(self, save_dir: Path, load_dir: Path = None, max_keep: int = 5):
        self.save_dir = Path(save_dir)
        self.load_dir = Path(load_dir) if load_dir else self.save_dir
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.max_keep = max_keep
        self.best_score = -1
        self.best_epoch = -1
        self.history = []
    
    def save(self, model, optimizer, scheduler, scaler, epoch: int, metrics: dict):
        ckpt = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'metrics': metrics,
            'best_score': self.best_score,
            'best_epoch': self.best_epoch,
            'config': {
                'features': cfg.FEATURES,
                'n_blocks': cfg.N_BLOCKS,
                'patch_size': cfg.PATCH_SIZE,
            }
        }
        
        torch.save(ckpt, self.save_dir / 'last_checkpoint.pth')
        
        val_dice = metrics.get('val_dice', 0)
        if val_dice > 0 and val_dice > self.best_score:
            self.best_score = val_dice
            self.best_epoch = epoch
            ckpt['best_score'] = self.best_score
            ckpt['best_epoch'] = self.best_epoch
            torch.save(ckpt, self.save_dir / 'best_model.pth')
            print(f"  >>> New best model! Val Dice: {val_dice:.4f}")
        
        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            torch.save(ckpt, self.save_dir / f'checkpoint_epoch_{epoch+1}.pth')
            self._cleanup_old_checkpoints()
        
        self.history.append({'epoch': epoch, **metrics})
        with open(self.save_dir / 'history.json', 'w') as f:
            json.dump(self.history, f, indent=2)
    
    def load(self, model, optimizer=None, scheduler=None, scaler=None,
             checkpoint_path=None, load_best: bool = False) -> int:
        if checkpoint_path is None:
            if load_best:
                checkpoint_path = self.load_dir / 'best_model.pth'
            else:
                checkpoint_path = self.load_dir / 'last_checkpoint.pth'
        
        checkpoint_path = Path(checkpoint_path)
        if not checkpoint_path.exists():
            print("No checkpoint found, starting fresh training")
            return 0
        
        print(f"Loading checkpoint from {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=cfg.DEVICE, weights_only=False)
        
        model.load_state_dict(ckpt['model_state_dict'])
        if optimizer and 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if scheduler and ckpt.get('scheduler_state_dict'):
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if scaler and ckpt.get('scaler_state_dict'):
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        
        self.best_score = ckpt.get('best_score', -1)
        self.best_epoch = ckpt.get('best_epoch', -1)
        
        history_path = self.save_dir / 'history.json'
        if history_path.exists():
            with open(history_path, 'r') as f:
                self.history = json.load(f)
        
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {ckpt['epoch']}")
        if self.best_score > 0:
            print(f"Best score so far: {self.best_score:.4f} at epoch {self.best_epoch}")
        return start_epoch
    
    def _cleanup_old_checkpoints(self):
        checkpoints = sorted(self.save_dir.glob('checkpoint_epoch_*.pth'))
        while len(checkpoints) > self.max_keep:
            checkpoints[0].unlink()
            checkpoints = checkpoints[1:]

print("CheckpointManager defined")

In [ ]:
# =============================================================================
# CELL 3: DATASET WITH FULL nnUNet AUGMENTATION PIPELINE
# =============================================================================

def load_volume_fast(path: Path) -> np.ndarray:
    """Load 3D TIF volume."""
    return tifffile.imread(str(path))


# ---- nnUNet Augmentation Functions ----

def augment_spatial(image, label, cfg):
    """
    nnUNet spatial augmentation: random rotation + scaling.
    Applied via scipy affine transform for 3D volumes.
    """
    do_rotation = random.random() < cfg.AUG_ROTATION_P
    do_scale = random.random() < cfg.AUG_SCALE_P
    
    if not do_rotation and not do_scale:
        return image, label
    
    # Build rotation matrix
    center = np.array(image.shape) / 2.0
    
    # Random rotation angles for each axis pair
    rot_range = cfg.AUG_ROTATION_RANGE * np.pi / 180.0
    
    if do_rotation:
        # Random angle for one random axis pair
        angle = random.uniform(-rot_range, rot_range)
        axis_pair = random.choice([(0, 1), (0, 2), (1, 2)])
    else:
        angle = 0
        axis_pair = (0, 1)
    
    # Scale factor
    if do_scale:
        scale = random.uniform(*cfg.AUG_SCALE_RANGE)
    else:
        scale = 1.0
    
    # Build 3x3 rotation matrix
    R = np.eye(3)
    c, s = np.cos(angle), np.sin(angle)
    i, j = axis_pair
    R[i, i] = c
    R[i, j] = -s
    R[j, i] = s
    R[j, j] = c
    
    # Apply scale
    R = R / scale
    
    # Compute offset so rotation is around center
    offset = center - R @ center
    
    # Apply transform
    image_out = ndimage.affine_transform(
        image, R, offset=offset, order=3, mode='reflect'
    )
    label_out = ndimage.affine_transform(
        label.astype(np.float32), R, offset=offset, order=0, mode='constant', cval=2
    ).astype(np.uint8)
    
    return image_out, label_out


def augment_gaussian_noise(image, cfg):
    """nnUNet: additive Gaussian noise."""
    if random.random() >= cfg.AUG_GAUSSIAN_NOISE_P:
        return image
    std = random.uniform(*cfg.AUG_GAUSSIAN_NOISE_STD)
    noise = np.random.normal(0, std, image.shape).astype(np.float32)
    return image + noise


def augment_gaussian_blur(image, cfg):
    """nnUNet: Gaussian blur."""
    if random.random() >= cfg.AUG_GAUSSIAN_BLUR_P:
        return image
    sigma = random.uniform(*cfg.AUG_GAUSSIAN_BLUR_SIGMA)
    return gaussian_filter(image, sigma=sigma).astype(np.float32)


def augment_brightness(image, cfg):
    """nnUNet: brightness augmentation (additive)."""
    if random.random() >= cfg.AUG_BRIGHTNESS_P:
        return image
    offset = random.uniform(-cfg.AUG_BRIGHTNESS_RANGE, cfg.AUG_BRIGHTNESS_RANGE)
    return image + offset


def augment_contrast(image, cfg):
    """nnUNet: contrast augmentation (multiplicative)."""
    if random.random() >= cfg.AUG_CONTRAST_P:
        return image
    factor = random.uniform(*cfg.AUG_CONTRAST_RANGE)
    mean = image.mean()
    return (image - mean) * factor + mean


def augment_gamma(image, cfg):
    """nnUNet: gamma augmentation."""
    # Normal gamma
    if random.random() < cfg.AUG_GAMMA_P:
        gamma = random.uniform(*cfg.AUG_GAMMA_RANGE)
        mn, mx = image.min(), image.max()
        rng = mx - mn + 1e-8
        image = ((image - mn) / rng) ** gamma * rng + mn
    
    # Inverted gamma (rare)
    if random.random() < cfg.AUG_GAMMA_INVERT_P:
        gamma = random.uniform(*cfg.AUG_GAMMA_RANGE)
        mn, mx = image.min(), image.max()
        rng = mx - mn + 1e-8
        image = rng - ((rng - (image - mn)) / rng) ** gamma * rng + mn
    
    return image


def augment_simulate_low_resolution(image, cfg):
    """nnUNet: simulate low resolution per-axis."""
    if random.random() >= cfg.AUG_LOW_RES_P:
        return image
    
    # Pick a random axis to downsample
    axis = random.randint(0, 2)
    zoom_factor = random.uniform(*cfg.AUG_LOW_RES_ZOOM)
    
    if zoom_factor >= 0.99:
        return image
    
    shape = list(image.shape)
    zoom = [1.0, 1.0, 1.0]
    zoom[axis] = zoom_factor
    
    # Downsample then upsample
    downsampled = ndimage.zoom(image, zoom, order=0)
    zoom_back = [1.0, 1.0, 1.0]
    zoom_back[axis] = shape[axis] / downsampled.shape[axis]
    upsampled = ndimage.zoom(downsampled, zoom_back, order=3)
    
    # Handle slight shape mismatches from rounding
    if upsampled.shape != tuple(shape):
        result = np.zeros(shape, dtype=np.float32)
        slices = tuple(slice(0, min(s, u)) for s, u in zip(shape, upsampled.shape))
        result[slices] = upsampled[slices]
        return result
    
    return upsampled.astype(np.float32)


def augment_mirror(image, label, cfg):
    """nnUNet: mirror/flip along each axis independently."""
    for axis in range(3):
        if random.random() < cfg.AUG_MIRROR_P:
            image = np.flip(image, axis)
            label = np.flip(label, axis)
    return np.ascontiguousarray(image), np.ascontiguousarray(label)


def apply_nnunet_augmentations(image, label, cfg):
    """
    Full nnUNet augmentation pipeline applied in order.
    Order matters: spatial first, then intensity, then mirror.
    """
    # 1. Spatial: rotation + scaling
    image, label = augment_spatial(image, label, cfg)
    
    # 2. Gaussian noise
    image = augment_gaussian_noise(image, cfg)
    
    # 3. Gaussian blur
    image = augment_gaussian_blur(image, cfg)
    
    # 4. Brightness
    image = augment_brightness(image, cfg)
    
    # 5. Contrast
    image = augment_contrast(image, cfg)
    
    # 6. Gamma
    image = augment_gamma(image, cfg)
    
    # 7. Simulate low resolution
    image = augment_simulate_low_resolution(image, cfg)
    
    # 8. Mirror (flips)
    image, label = augment_mirror(image, label, cfg)
    
    return image.astype(np.float32), label


class VesuviusDatasetV3(Dataset):
    """
    V3 Dataset with full nnUNet augmentation pipeline.
    Preloads all data to RAM.
    Uses nnUNet-style foreground oversampling (33%).
    """
    
    def __init__(
        self,
        csv_path: Path,
        images_dir: Path,
        labels_dir: Path,
        patch_size: Tuple[int, int, int] = (128, 128, 128),
        is_train: bool = True,
        data_fraction: float = 1.0,
        patches_per_volume: int = 12,
        preload: bool = True,
    ):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        
        df = pd.read_csv(csv_path)
        valid_ids = []
        for idx in df['id'].values:
            if (self.images_dir / f"{idx}.tif").exists() and \
               (self.labels_dir / f"{idx}.tif").exists():
                valid_ids.append(idx)
        
        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.shuffle(valid_ids)
            valid_ids = valid_ids[:n]
        
        self.volume_ids = valid_ids
        self.volumes = {}
        self.fg_coords = {}
        
        if preload:
            print(f"Preloading {len(self.volume_ids)} volumes to RAM...")
            for vol_id in tqdm(self.volume_ids, desc="Loading"):
                img = load_volume_fast(self.images_dir / f"{vol_id}.tif").astype(np.float32)
                lbl = load_volume_fast(self.labels_dir / f"{vol_id}.tif").astype(np.uint8)
                
                # Per-volume z-score normalization
                img = (img - img.mean()) / (img.std() + 1e-8)
                
                self.volumes[vol_id] = (img, lbl)
                
                # Cache foreground coordinates for oversampling
                fg = np.argwhere(lbl == 1)
                if len(fg) > 10000:
                    fg = fg[np.random.choice(len(fg), 10000, replace=False)]
                self.fg_coords[vol_id] = fg if len(fg) > 0 else None
            
            sample_vol = next(iter(self.volumes.values()))
            vol_size_mb = (sample_vol[0].nbytes + sample_vol[1].nbytes) / 1e6
            total_gb = vol_size_mb * len(self.volumes) / 1000
            print(f"Loaded {len(self.volumes)} volumes ({total_gb:.1f} GB in RAM)")
        
        print(f"Dataset ready: {len(self)} samples")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def __getitem__(self, idx):
        vol_idx = idx // self.patches_per_volume
        vol_id = self.volume_ids[vol_idx]
        
        image, label = self.volumes[vol_id]
        d, h, w = image.shape
        pd, ph, pw = self.patch_size
        
        # Pad if needed
        if d < pd or h < ph or w < pw:
            pad_d = max(0, pd - d)
            pad_h = max(0, ph - h)
            pad_w = max(0, pw - w)
            image = np.pad(image, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='reflect')
            label = np.pad(label, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='constant', constant_values=2)
            d, h, w = image.shape
        
        # nnUNet-style foreground oversampling: 33% force foreground center
        fg = self.fg_coords.get(vol_id)
        force_fg = self.is_train and random.random() < cfg.FG_OVERSAMPLE_RATIO and fg is not None and len(fg) > 0
        
        if force_fg:
            c = fg[random.randint(0, len(fg) - 1)]
            z = max(0, min(c[0] - pd // 2, d - pd))
            y = max(0, min(c[1] - ph // 2, h - ph))
            x = max(0, min(c[2] - pw // 2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))
        
        img_patch = image[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_patch = label[z:z+pd, y:y+ph, x:x+pw].copy()
        
        # Apply full nnUNet augmentation pipeline
        if self.is_train:
            img_patch, lbl_patch = apply_nnunet_augmentations(img_patch, lbl_patch, cfg)
        
        fg_mask = (lbl_patch == 1).astype(np.float32)
        ig_mask = (lbl_patch == 2).astype(np.float32)
        
        return {
            'image': torch.from_numpy(img_patch).unsqueeze(0).float(),
            'label': torch.from_numpy(fg_mask).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ig_mask).unsqueeze(0).float(),
        }


print("V3 Dataset with nnUNet augmentation pipeline ready")
print("Augmentations: rotation, scale, noise, blur, brightness,")
print("  contrast, gamma, simulate low-res, mirror")

In [ ]:
# =============================================================================
# CELL 4: MODEL - nnUNet-Style ResEncUNet3D with InstanceNorm
# =============================================================================

class ConvBlock(nn.Module):
    """3D convolution + InstanceNorm3d (affine=True) + LeakyReLU.
    Matches host baseline: InstanceNorm with affine, conv_bias=True.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=True),  # bias=True like host
            nn.InstanceNorm3d(out_ch, affine=True),             # InstanceNorm like host
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    """Residual block with N conv-norm-relu sequences.
    nnUNet uses pre-activation style but we keep post-activation for compatibility.
    """
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ConvBlock(channels, channels) for _ in range(n_convs)]
        )
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    """Concurrent Spatial and Channel Squeeze-and-Excitation."""
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    """
    6-stage Residual Encoder U-Net with:
    - InstanceNorm3d (affine=True) matching host baseline
    - conv_bias=True matching host baseline
    - scSE attention (extra over host)
    - Deep supervision with nnUNet-style weights
    - Decoder: single ConvBlock per stage (closer to host)
    """
    
    def __init__(
        self,
        in_ch: int = 1,
        out_ch: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_deep_supervision: bool = True,
    ):
        super().__init__()
        
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.n_stages = len(features)
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat, n_convs=2) for _ in range(nb)]
            )
            self.encoders.append(encoder)
            
            if use_scse:
                self.attentions.append(scSEBlock(feat))
            else:
                self.attentions.append(nn.Identity())
            
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2, bias=True))
        
        # Decoder - single ConvBlock per stage (closer to nnUNet host)
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2, bias=True))
            self.dec_convs.append(ConvBlock(out_feat * 2, out_feat))
        
        self.final = nn.Conv3d(features[0], out_ch, 1, bias=True)
        
        # Deep supervision heads
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            # nnUNet-style: outputs at each decoder stage except bottleneck
            # weights: [1, 0.5, 0.25, 0.125, ...] normalized
            n_ds_outputs = min(4, len(features) - 1)  # More DS outputs than V2
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1, bias=True))
    
    def forward(self, x):
        enc_features = []
        
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        
        ds_outputs = []
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final(x)
        
        if self.use_deep_supervision and self.training:
            return {'output': out, 'deep': ds_outputs}
        return out


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Quick test
print("nnUNet-style ResEncUNet3D defined")
print("Key changes from V2:")
print("  - InstanceNorm3d (affine=True) instead of GroupNorm")
print("  - conv_bias=True")
print("  - 4 deep supervision outputs (was 3)")

In [ ]:
# =============================================================================
# CELL 5: LOSS FUNCTIONS
# Dice + CrossEntropy (not BCE) + MedialSurfaceRecall + soft-clDice
# =============================================================================

class DiceLoss(nn.Module):
    """Per-sample Dice loss (nnUNet default: batch_dice=False).
    Uses sigmoid internally.
    """
    def __init__(self, smooth: float = 1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        # Per-sample dice (nnUNet default)
        batch_size = pred.shape[0]
        dice_sum = 0.0
        for b in range(batch_size):
            p = pred[b].reshape(-1)
            t = target[b].reshape(-1)
            intersection = (p * t).sum()
            dice_sum += (2 * intersection + self.smooth) / (p.sum() + t.sum() + self.smooth)
        
        return 1 - dice_sum / batch_size


class CEWithMask(nn.Module):
    """CrossEntropy loss with ignore mask support.
    Host baseline uses DC + CE (CrossEntropy, not BCE).
    For binary segmentation with sigmoid output, this is equivalent
    to BCE but we use the CE formulation for compatibility.
    """
    def forward(self, pred, target, mask=None):
        if mask is not None:
            valid = 1 - mask
            loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
            loss = (loss * valid).sum() / (valid.sum() + 1e-8)
        else:
            loss = F.binary_cross_entropy_with_logits(pred, target)
        return loss


def soft_skeletonize(x, num_iter=40):
    """Soft skeletonization using iterative min-max pooling."""
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    return x


class MedialSurfaceRecallLoss(nn.Module):
    """
    Medial Surface Recall - host baseline's skeleton loss.
    Computes 2D skeletonization per-slice across all 3 axes,
    then measures recall of the medial surface.
    
    This is more appropriate for sheet-like structures (papyrus)
    than 3D skeletonization which finds centerlines.
    """
    def __init__(self, smooth: float = 1e-5):
        super().__init__()
        self.smooth = smooth
    
    def _compute_medial_surface(self, binary_volume):
        """
        Compute medial surface by 2D skeletonization along each axis.
        Returns union of all per-slice skeletons.
        """
        vol_np = binary_volume.detach().cpu().numpy()
        medial = np.zeros_like(vol_np, dtype=bool)
        
        # Skeletonize along each axis
        for axis in range(3):
            for i in range(vol_np.shape[axis]):
                if axis == 0:
                    sl = vol_np[i, :, :]
                elif axis == 1:
                    sl = vol_np[:, i, :]
                else:
                    sl = vol_np[:, :, i]
                
                if sl.sum() > 0:
                    skel = skeletonize(sl.astype(bool))
                    if axis == 0:
                        medial[i, :, :] |= skel
                    elif axis == 1:
                        medial[:, i, :] |= skel
                    else:
                        medial[:, :, i] |= skel
        
        return medial
    
    def forward(self, pred, target, mask=None):
        pred_sigmoid = torch.sigmoid(pred)
        device = pred.device
        
        if mask is not None:
            pred_sigmoid = pred_sigmoid * (1 - mask)
            target = target * (1 - mask)
        
        batch_size = pred.shape[0]
        recall_sum = 0.0
        
        for b in range(batch_size):
            target_vol = target[b, 0]  # (D, H, W)
            pred_vol = pred_sigmoid[b, 0]
            
            # Binary target for skeletonization
            target_binary = (target_vol > 0.5).float()
            
            if target_binary.sum() < 10:
                continue
            
            # Compute medial surface of target
            medial_np = self._compute_medial_surface(target_binary)
            medial = torch.from_numpy(medial_np.astype(np.float32)).to(device)
            
            if medial.sum() < 1:
                continue
            
            # Recall: how well does prediction cover the medial surface?
            recall = (pred_vol * medial).sum() / (medial.sum() + self.smooth)
            recall_sum += recall
        
        if batch_size == 0:
            return torch.tensor(0.0, device=device, requires_grad=True)
        
        return 1 - recall_sum / batch_size


class SkeletonRecallLoss(nn.Module):
    """Fast soft skeleton recall (fallback when MedialSurface is too slow)."""
    def __init__(self, smooth: float = 1e-5, tube_radius: int = 2):
        super().__init__()
        self.smooth = smooth
        self.tube_radius = tube_radius
    
    def forward(self, pred, target, mask=None):
        pred_sigmoid = torch.sigmoid(pred)
        
        if mask is not None:
            pred_sigmoid = pred_sigmoid * (1 - mask)
            target = target * (1 - mask)
        
        target_skel = soft_skeletonize(target, num_iter=10)
        
        if self.tube_radius > 0:
            target_tube = F.max_pool3d(
                target_skel,
                kernel_size=2 * self.tube_radius + 1,
                stride=1,
                padding=self.tube_radius
            )
        else:
            target_tube = target_skel
        
        recall = (pred_sigmoid * target_tube).sum() / (target_tube.sum() + self.smooth)
        return 1 - recall


class SoftClDiceLoss(nn.Module):
    """Soft clDice (centerline Dice) loss."""
    def __init__(self, smooth: float = 1e-5, num_iter: int = 10):
        super().__init__()
        self.smooth = smooth
        self.num_iter = num_iter
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        skel_pred = soft_skeletonize(pred, self.num_iter)
        skel_target = soft_skeletonize(target, self.num_iter)
        
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_target * pred).sum() + self.smooth) / (skel_target.sum() + self.smooth)
        cldice = 2 * tprec * tsens / (tprec + tsens + self.smooth)
        
        return 1 - cldice


class CombinedLossV3(nn.Module):
    """
    V3 Combined loss:
    - Dice + CE (matching host baseline)
    - MedialSurfaceRecall OR fast SkeletonRecall
    - soft-clDice
    - Deep supervision with nnUNet-style exponential weights
    
    Set use_medial_surface=True for host-matching loss.
    Set use_medial_surface=False for faster training with soft skeleton.
    """
    def __init__(
        self,
        dice_weight: float = 0.5,
        ce_weight: float = 0.5,
        skeleton_weight: float = 0.3,
        cldice_weight: float = 0.2,
        skeleton_start: int = 200,
        skeleton_warmup: int = 200,
        cldice_start: int = 400,
        cldice_warmup: int = 200,
        use_medial_surface: bool = False,  # True = host-matching but slower
    ):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight
        self.skeleton_weight = skeleton_weight
        self.cldice_weight = cldice_weight
        
        self.skeleton_start = skeleton_start
        self.skeleton_warmup = skeleton_warmup
        self.cldice_start = cldice_start
        self.cldice_warmup = cldice_warmup
        
        # nnUNet-style deep supervision weights: exponential decay
        # [1, 0.5, 0.25, 0.125] normalized
        raw_weights = [1.0 / (2 ** i) for i in range(4)]
        total = sum(raw_weights)
        self.ds_weights = [w / total for w in raw_weights]
        
        self.dice_loss = DiceLoss()
        self.ce_loss = CEWithMask()
        
        if use_medial_surface:
            self.skeleton_loss = MedialSurfaceRecallLoss()
        else:
            self.skeleton_loss = SkeletonRecallLoss()
        
        self.cldice_loss = SoftClDiceLoss()
    
    def _get_scale(self, epoch, start, warmup):
        if epoch < start:
            return 0.0
        elif epoch < start + warmup:
            return (epoch - start) / warmup
        return 1.0
    
    def forward(self, output, target, ignore_mask, epoch):
        if isinstance(output, dict):
            pred = output['output']
            deep_outputs = output.get('deep', [])
        else:
            pred = output
            deep_outputs = []
        
        skel_scale = self._get_scale(epoch, self.skeleton_start, self.skeleton_warmup)
        cldice_scale = self._get_scale(epoch, self.cldice_start, self.cldice_warmup)
        
        # Base losses: Dice + CE (host baseline combination)
        dice = self.dice_loss(pred, target, ignore_mask)
        ce = self.ce_loss(pred, target, ignore_mask)
        
        # Topology losses
        if skel_scale > 0:
            skel = self.skeleton_loss(pred, target, ignore_mask)
        else:
            skel = torch.tensor(0.0, device=pred.device)
        
        if cldice_scale > 0:
            cldice = self.cldice_loss(pred, target, ignore_mask)
        else:
            cldice = torch.tensor(0.0, device=pred.device)
        
        # Combine: base + scaled topology
        total = (
            self.dice_weight * dice +
            self.ce_weight * ce +
            self.skeleton_weight * skel_scale * skel +
            self.cldice_weight * cldice_scale * cldice
        )
        
        # Deep supervision with nnUNet-style exponential weights
        for i, ds_out in enumerate(deep_outputs):
            if i >= len(self.ds_weights):
                break
            ds_target = F.interpolate(target, size=ds_out.shape[2:], mode='nearest')
            ds_mask = F.interpolate(ignore_mask, size=ds_out.shape[2:], mode='nearest')
            ds_dice = self.dice_loss(ds_out, ds_target, ds_mask)
            ds_ce = self.ce_loss(ds_out, ds_target, ds_mask)
            # Weight DS outputs with same dice/ce ratio
            ds_loss = self.dice_weight * ds_dice + self.ce_weight * ds_ce
            total = total + self.ds_weights[i] * ds_loss
        
        return {
            'total': total,
            'dice': dice.item(),
            'ce': ce.item(),
            'skeleton': skel.item() if skel_scale > 0 else 0.0,
            'cldice': cldice.item() if cldice_scale > 0 else 0.0,
            'skel_scale': skel_scale,
            'cldice_scale': cldice_scale,
        }


print("V3 Loss functions defined")
print(f"Base: Dice ({cfg.DICE_WEIGHT}) + CE ({cfg.CE_WEIGHT})")
print(f"Topology: SkeletonRecall ({cfg.SKELETON_WEIGHT}) + clDice ({cfg.CLDICE_WEIGHT})")
print(f"Schedule: Skeleton @ epoch {cfg.SKELETON_START_EPOCH}, clDice @ epoch {cfg.CLDICE_START_EPOCH}")
print(f"DS weights (nnUNet-style): {[f'{w:.3f}' for w in [1/(2**i) for i in range(4)]]}")

In [ ]:
# =============================================================================
# CELL 6: TRAINING FUNCTIONS
# =============================================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    """Train one epoch with full logging."""
    model.train()
    
    total_loss = 0
    total_dice = 0
    total_ce = 0
    total_skel = 0
    total_cldice = 0
    n_batches = 0
    
    pbar = tqdm(total=len(loader), desc=f"Epoch {epoch+1}",
                file=sys.stdout, dynamic_ncols=True)
    
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        ignore = batch['ignore_mask'].to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            losses = criterion(output, labels, ignore, epoch)
        
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += losses['total'].item()
        total_dice += losses['dice']
        total_ce += losses['ce']
        total_skel += losses.get('skeleton', 0)
        total_cldice += losses.get('cldice', 0)
        n_batches += 1
        
        pbar.update(1)
        pbar.set_postfix({
            'loss': f"{losses['total'].item():.3f}",
            'dice': f"{losses['dice']:.3f}",
            'ce': f"{losses['ce']:.3f}",
        })
        
        if n_batches % 50 == 0:
            sys.stdout.flush()
    
    pbar.close()
    sys.stdout.flush()
    
    return {
        'train_loss': total_loss / n_batches,
        'train_dice_loss': total_dice / n_batches,
        'train_ce_loss': total_ce / n_batches,
        'train_skel_loss': total_skel / n_batches,
        'train_cldice_loss': total_cldice / n_batches,
    }


@torch.no_grad()
def validate(model, loader, device):
    """Validation with per-sample Dice."""
    model.eval()
    
    total_dice = 0
    n_samples = 0
    
    for batch in tqdm(loader, desc="Val", file=sys.stdout, leave=False):
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].numpy()
        ignore = batch['ignore_mask'].numpy()
        
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            if isinstance(output, dict):
                output = output['output']
            probs = torch.sigmoid(output).cpu().numpy()
        
        for b in range(images.shape[0]):
            pred = (probs[b, 0] > 0.5).astype(np.uint8)
            tgt = labels[b, 0].astype(np.uint8)
            ign = ignore[b, 0] > 0.5
            pred[ign] = 0
            tgt[ign] = 0
            
            inter = (pred & tgt).sum()
            union = pred.sum() + tgt.sum()
            total_dice += (2 * inter + 1e-5) / (union + 1e-5)
            n_samples += 1
    
    sys.stdout.flush()
    return {'val_dice': total_dice / max(n_samples, 1)}


print("Training functions ready")

In [ ]:
# =============================================================================
# CELL 7: MAIN TRAINING LOOP (OPTIMIZED FOR H100)
# =============================================================================

def main_training():
    """V3 nnUNet-style training for H100 + 220GB RAM."""
    
    print("="*60)
    print("VESUVIUS V3 nnUnet-STYLE TRAINING (H100 OPTIMIZED)")
    print("="*60)
    print(f"Patch size: {cfg.PATCH_SIZE}")
    print(f"Batch size: {cfg.BATCH_SIZE}")  # CRITICAL: Increased to 8
    print(f"Loss: Dice + CE + SkeletonRecall + clDice")
    print(f"Norm: InstanceNorm3d")
    print("="*60)
    import multiprocessing
    #multiprocessing.set_start_method('fork', force=True) # Use 'spawn' if fork causes issues, but it's usually slower
    
    # Paths
    train_csv = cfg.DATA_ROOT / "train.csv"
    train_images = cfg.DATA_ROOT / "train_images"
    train_labels = cfg.DATA_ROOT / "train_labels"
    
    # Create datasets
    print("\n[1/4] Loading training data to RAM...")
    train_dataset = VesuviusDatasetV3(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=True,
        data_fraction=cfg.DATA_FRACTION,
        patches_per_volume=cfg.PATCHES_PER_VOLUME,
        preload=True,
    )
    
    print("\n[2/4] Loading validation data to RAM...")
    val_dataset = VesuviusDatasetV3(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=False,
        data_fraction=0.15,
        patches_per_volume=1,
        preload=True,
    )
    
    # OPTIMIZED DataLoaders for H100
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,  # Increased to 12
        pin_memory=True,
        drop_last=True,
        persistent_workers=True,
        prefetch_factor=16,  # Increased from 4 to 8 for better GPU utilization
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=12,  # Increased for validation
        pin_memory=True,
        persistent_workers=True,
    )
    
    print(f"\nTrain: {len(train_dataset)} samples, {len(train_loader)} batches/epoch")
    print(f"Val: {len(val_dataset)} samples")
    
    # Model
    print("\n[3/4] Creating model...")
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    ).to(cfg.DEVICE)
    
    # CRITICAL: Compile model for H100 optimization
    if hasattr(torch, 'compile') and torch.cuda.is_available():
        print("Compiling model with torch.compile() for H100 optimization...")
        model = torch.compile(model, mode='reduce-overhead', fullgraph=True)
    
    print(f"Parameters: {count_parameters(model):,}")
    
    # Loss
    criterion = CombinedLossV3(
        dice_weight=cfg.DICE_WEIGHT,
        ce_weight=cfg.CE_WEIGHT,
        skeleton_weight=cfg.SKELETON_WEIGHT,
        cldice_weight=cfg.CLDICE_WEIGHT,
        skeleton_start=cfg.SKELETON_START_EPOCH,
        skeleton_warmup=cfg.SKELETON_WARMUP_EPOCHS,
        cldice_start=cfg.CLDICE_START_EPOCH,
        cldice_warmup=cfg.CLDICE_WARMUP_EPOCHS,
        use_medial_surface=False,  # Set True for host-matching (slower)
    )
    
    # Optimizer - SGD with Nesterov (matching host baseline)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=cfg.LEARNING_RATE,  # Increased to 0.028 for larger batch
        momentum=cfg.MOMENTUM,
        weight_decay=cfg.WEIGHT_DECAY,
        nesterov=True,
    )
    
    scaler = GradScaler(enabled=cfg.USE_AMP)
    
    # Checkpoint manager
    ckpt_mgr = CheckpointManager(
        save_dir=cfg.CHECKPOINT_DIR,
    )
    
    # Resume
    print("\n[4/4] Checking for checkpoint...")
    if cfg.RESUME_TRAINING:
        start_epoch = ckpt_mgr.load(model, optimizer, None, scaler)
    else:
        start_epoch = 0
    
    # LR override
    if cfg.OVERRIDE_LR > 0 and start_epoch > 0:
        old_lr = optimizer.param_groups[0]['lr']
        for param_group in optimizer.param_groups:
            param_group['lr'] = cfg.OVERRIDE_LR
            param_group['initial_lr'] = cfg.OVERRIDE_LR
        print(f"\n>>> LR OVERRIDE: {old_lr:.6f} -> {cfg.OVERRIDE_LR:.6f}")
        remaining_epochs = cfg.EPOCHS - start_epoch
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=remaining_epochs, eta_min=1e-6,
        )
    else:
        # nnUNet polynomial LR: (1 - epoch/total)^0.9
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer, lambda e: (1 - e / cfg.EPOCHS) ** 0.9
        )
        if start_epoch > 0:
            for _ in range(start_epoch):
                scheduler.step()
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\nStarting from epoch {start_epoch + 1}")
    print(f"Current LR: {current_lr:.6f}")
    print("="*60)
    print("TRAINING STARTED")
    print("="*60)
    
    for epoch in range(start_epoch, cfg.EPOCHS):
        epoch_start = time.time()
        
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, cfg.DEVICE, epoch
        )
        
        scheduler.step()
        
        if (epoch + 1) % cfg.VALIDATE_EVERY == 0:
            val_metrics = validate(model, val_loader, cfg.DEVICE)
        else:
            val_metrics = {'val_dice': 0}
        
        epoch_time = time.time() - epoch_start
        lr = optimizer.param_groups[0]['lr']
        
        log_parts = [
            f"Epoch {epoch+1}/{cfg.EPOCHS}",
            f"{epoch_time:.1f}s",
            f"LR: {lr:.6f}",
            f"Loss: {train_metrics['train_loss']:.4f}",
            f"Dice: {train_metrics['train_dice_loss']:.4f}",
            f"CE: {train_metrics['train_ce_loss']:.4f}",
        ]
        if train_metrics['train_skel_loss'] > 0:
            log_parts.append(f"Skel: {train_metrics['train_skel_loss']:.4f}")
        if train_metrics['train_cldice_loss'] > 0:
            log_parts.append(f"clD: {train_metrics['train_cldice_loss']:.4f}")
        if val_metrics['val_dice'] > 0:
            log_parts.append(f"Val: {val_metrics['val_dice']:.4f}")
        
        print(" | ".join(log_parts))
        
        ckpt_mgr.save(model, optimizer, scheduler, scaler, epoch,
                      {**train_metrics, **val_metrics})
    
    print("\n" + "="*60)
    print(f"DONE! Best: {ckpt_mgr.best_score:.4f} @ epoch {ckpt_mgr.best_epoch}")
    print("="*60)
    
    return model, ckpt_mgr

print("Ready to train with H100 optimizations!")

In [ ]:
# =============================================================================
# CELL 8: RUN TRAINING
# =============================================================================

# CONFIGURATION
cfg.RESUME_TRAINING = False  # Fresh V3 training
cfg.DATA_FRACTION = 1.0
cfg.EPOCHS = 1200
cfg.VALIDATE_EVERY = 5

# START TRAINING
model, ckpt_mgr = main_training()

In [ ]:
# =============================================================================
# CELL 9: POST-PROCESSING FUNCTIONS
# =============================================================================

def connected_components_3d(volume, connectivity=26):
    if USE_CC3D:
        return cc3d.connected_components(volume, connectivity=connectivity)
    else:
        if connectivity == 26:
            struct = ndimage.generate_binary_structure(3, 3)
        elif connectivity == 6:
            struct = ndimage.generate_binary_structure(3, 1)
        else:
            struct = ndimage.generate_binary_structure(3, 2)
        labeled, _ = ndimage.label(volume, structure=struct)
        return labeled


def simple_postprocess(
    pred_prob: np.ndarray,
    threshold: float = 0.60,
    min_component_size: int = 25,
) -> np.ndarray:
    """Simple threshold + small component removal. Proven to work well."""
    binary = (pred_prob > threshold).astype(np.uint8)
    
    if min_component_size > 0:
        labeled = connected_components_3d(binary)
        for i in range(1, labeled.max() + 1):
            if (labeled == i).sum() < min_component_size:
                binary[labeled == i] = 0
    
    return binary


print("Post-processing functions defined")

In [ ]:
# =============================================================================
# CELL 10: INFERENCE FUNCTIONS
# =============================================================================

@torch.no_grad()
def sliding_window_inference(
    model,
    volume: np.ndarray,
    patch_size: Tuple[int, int, int] = (128, 128, 128),
    overlap: float = 0.5,
    device: str = 'cuda',
) -> np.ndarray:
    """Sliding window inference with Gaussian weighting."""
    model.eval()
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd * (1 - overlap)), int(ph * (1 - overlap)), int(pw * (1 - overlap))
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)
    
    sigma = 0.125
    gauss_z = np.exp(-0.5 * ((np.arange(pd) - pd/2) / (pd * sigma)) ** 2)
    gauss_y = np.exp(-0.5 * ((np.arange(ph) - ph/2) / (ph * sigma)) ** 2)
    gauss_x = np.exp(-0.5 * ((np.arange(pw) - pw/2) / (pw * sigma)) ** 2)
    gauss_weight = gauss_z[:, None, None] * gauss_y[None, :, None] * gauss_x[None, None, :]
    gauss_weight = gauss_weight.astype(np.float32)
    
    z_pos = list(set(list(range(0, max(1, D-pd+1), sd)) + ([D-pd] if D > pd else [0])))
    y_pos = list(set(list(range(0, max(1, H-ph+1), sh)) + ([H-ph] if H > ph else [0])))
    x_pos = list(set(list(range(0, max(1, W-pw+1), sw)) + ([W-pw] if W > pw else [0])))
    
    vol_norm = (volume - volume.mean()) / (volume.std() + 1e-8)
    
    for z in tqdm(z_pos, desc="Inference", leave=False):
        for y in y_pos:
            for x in x_pos:
                patch = vol_norm[z:z+pd, y:y+ph, x:x+pw].astype(np.float32)
                actual_shape = patch.shape
                
                if patch.shape != (pd, ph, pw):
                    pad = [(0, pd-patch.shape[0]), (0, ph-patch.shape[1]), (0, pw-patch.shape[2])]
                    patch = np.pad(patch, pad, mode='reflect')
                
                inp = torch.from_numpy(patch[None, None]).to(device)
                with autocast(enabled=cfg.USE_AMP):
                    out = model(inp)
                    if isinstance(out, dict):
                        out = out['output']
                    out = torch.sigmoid(out).squeeze().cpu().numpy()
                
                out = out[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                weight = gauss_weight[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                
                pred_sum[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += out * weight
                count[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += weight
    
    return pred_sum / np.maximum(count, 1e-8)


def inference_with_tta(
    model,
    volume: np.ndarray,
    patch_size: Tuple[int, int, int] = (128, 128, 128),
    overlap: float = 0.5,
    device: str = 'cuda',
    tta_level: str = 'flip',
) -> np.ndarray:
    """Inference with test-time augmentation."""
    preds = []
    
    pred = sliding_window_inference(model, volume, patch_size, overlap, device)
    preds.append(pred)
    
    if tta_level in ['flip', 'full']:
        for axis in [0, 1, 2]:
            vol_flip = np.flip(volume, axis).copy()
            pred_flip = sliding_window_inference(model, vol_flip, patch_size, overlap, device)
            preds.append(np.flip(pred_flip, axis))
    
    if tta_level == 'full':
        vol_rot = np.rot90(volume, k=1, axes=(1, 2)).copy()
        pred_rot = sliding_window_inference(model, vol_rot, patch_size, overlap, device)
        preds.append(np.rot90(pred_rot, k=-1, axes=(1, 2)))
    
    return np.mean(preds, axis=0)


print("Inference functions defined")

In [ ]:
# =============================================================================
# CELL 11: TRAINING STATUS CHECK
# =============================================================================

def check_training_status():
    last_ckpt = cfg.CHECKPOINT_DIR / 'last_checkpoint.pth'
    best_ckpt = cfg.CHECKPOINT_DIR / 'best_model.pth'
    history_file = cfg.CHECKPOINT_DIR / 'history.json'
    
    print("="*60)
    print("V3 TRAINING STATUS")
    print("="*60)
    
    if last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location='cpu', weights_only=False)
        print(f"\nLast checkpoint:")
        print(f"  Epoch: {ckpt['epoch'] + 1}/{cfg.EPOCHS}")
        print(f"  Progress: {100 * (ckpt['epoch'] + 1) / cfg.EPOCHS:.1f}%")
        if 'metrics' in ckpt:
            print(f"  Train Loss: {ckpt['metrics'].get('train_loss', 'N/A')}")
            print(f"  Val Dice: {ckpt['metrics'].get('val_dice', 'N/A')}")
    else:
        print("\nNo checkpoint found. Training not started.")
    
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
        print(f"\nBest model:")
        print(f"  Epoch: {ckpt['best_epoch'] + 1}")
        print(f"  Best Dice: {ckpt['best_score']:.4f}")
    
    if history_file.exists():
        with open(history_file, 'r') as f:
            history = json.load(f)
        print(f"\nTraining history: {len(history)} epochs recorded")
        if len(history) > 0:
            recent = history[-5:]
            print("\nRecent epochs:")
            for h in recent:
                print(f"  Epoch {h['epoch']+1}: loss={h.get('train_loss', 0):.4f}, "
                      f"val_dice={h.get('val_dice', 0):.4f}")
    
    print("="*60)

check_training_status()